# NBA Game Flow Analysis (Momentum & Runs)

バスケットボールの試合における「流れ（Momentum）」を可視化し、連続得点（Run）を抽出するツールです。
ロードマップに基づき、客観的なデータ（数字・事実）を提供することを目的としています。

In [ ]:
!pip install -q nba_api

In [ ]:
import pandas as pd
import time
import matplotlib.pyplot as plt
from nba_api.stats.endpoints import playbyplayv2

# カスタムヘッダーの設定（APIブロック回避のため）
custom_headers = {
    'Host': 'stats.nba.com',
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'application/json, text/plain, */*',
    'Accept-Language': 'en-US,en;q=0.5',
    'Referer': 'https://www.nba.com/',
    'Origin': 'https://www.nba.com',
    'Connection': 'keep-alive',
    'x-nba-stats-origin': 'stats',
    'x-nba-stats-token': 'true'
}

def analyze_game_flow(game_id, run_threshold=8):
    print(f"GameID: {game_id} のプレイ・バイ・プレイを解析中...")

    # PBPデータの取得
    try:
        # ヘッダーを指定してリクエスト
        pbp = playbyplayv2.PlayByPlayV2(game_id=game_id, headers=custom_headers, timeout=60)
        df = pbp.get_data_frames()[0]
    except Exception as e:
        return f"データ取得エラー: {e} - API接続に失敗しました。しばらく待って再試行してください。", None

    if df.empty:
        return "データ取得エラー: データが空です。GameIDが正しいか確認してください。", None

    # スコアが発生した行のみに絞り込む
    if 'SCORE' not in df.columns:
        return "データ解析エラー: 必要なカラム(SCORE)が見つかりません。", None

    df_scores = df[df['SCORE'].notnull()].copy()

    if df_scores.empty:
        return "データ解析エラー: スコアデータが見つかりません。", None

    # スコアを数値に変換 ("2 - 0" -> Visitor 2, Home 0)
    try:
        scores = df_scores['SCORE'].str.split(' - ', expand=True).astype(int)
        df_scores['V_SCORE'] = scores[0]
        df_scores['H_SCORE'] = scores[1]
    except Exception as e:
        return f"スコア解析エラー: {e}", None

    # どちらのチームが得点したかを判定するための差分計算
    df_scores['V_DIFF'] = df_scores['V_SCORE'].diff().fillna(df_scores['V_SCORE'])
    df_scores['H_DIFF'] = df_scores['H_SCORE'].diff().fillna(df_scores['H_SCORE'])

    runs = []
    current_run_pts = 0
    current_team = None
    
    # 連続得点（Run）の抽出
    for i, row in df_scores.iterrows():
        # 得点チームの特定
        if row['V_DIFF'] > 0:
            scoring_team = "VISITOR"
            pts = row['V_DIFF']
        elif row['H_DIFF'] > 0:
            scoring_team = "HOME"
            pts = row['H_DIFF']
        else:
            continue

        if scoring_team == current_team:
            current_run_pts += pts
        else:
            # 前のチームのランが閾値以上なら記録
            if current_run_pts >= run_threshold:
                runs.append({
                    'Quarter': row['PERIOD'],
                    'Time': row['PCTIMESTRING'],
                    'Run_Team': current_team,
                    'Run_Score': f"{int(current_run_pts)}-0",
                    'Score_At_End': row['SCORE']
                })
            # 新しいランの開始
            current_team = scoring_team
            current_run_pts = pts

    # 最後のランをチェック
    if current_run_pts >= run_threshold:
        runs.append({
            'Quarter': df_scores.iloc[-1]['PERIOD'],
            'Time': df_scores.iloc[-1]['PCTIMESTRING'],
            'Run_Team': current_team,
            'Run_Score': f"{int(current_run_pts)}-0",
            'Score_At_End': df_scores.iloc[-1]['SCORE']
        })

    return pd.DataFrame(runs), df_scores

def plot_pbp_flow(df, game_id):
    if df is None or df.empty:
        print("プロット用データがありません。")
        return

    # 点差 (Home - Visitor)
    df['CALC_MARGIN'] = df['H_SCORE'] - df['V_SCORE']

    plt.figure(figsize=(14, 6))
    
    x_range = range(len(df))
    
    # マージン推移
    plt.plot(x_range, df['CALC_MARGIN'], color='black', alpha=0.3)

    # Momentumの可視化 (リードしている側の色で塗りつぶし)
    plt.fill_between(x_range, df['CALC_MARGIN'], 0, where=(df['CALC_MARGIN'] >= 0), color='#006BB6', alpha=0.6, label='HOME Lead')
    plt.fill_between(x_range, df['CALC_MARGIN'], 0, where=(df['CALC_MARGIN'] < 0), color='#C9082A', alpha=0.6, label='VISITOR Lead')

    plt.axhline(y=0, color='gray', linestyle='-', linewidth=0.5)
    plt.title(f"Game Flow Momentum: {game_id} (Score Margin)")
    plt.ylabel("Score Margin (Home - Visitor)")
    plt.xlabel("Scoring Events")
    plt.legend()
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    plt.show()

In [ ]:
# 実行セクション

# GameIDの設定 (例: 2025 Summer League SAS vs NYK)
# 参照: https://www.nba.com/game/sas-vs-nyk-0062500001/play-by-play
target_game_id = '0062500001' 

# 解析実行
result = analyze_game_flow(target_game_id, run_threshold=8)

# 結果の表示
if isinstance(result[0], str):
    print(f"エラーが発生しました: {result[0]}")
    print("\n【ヒント】")
    print("API接続がタイムアウトする場合、時間をおいて再試行してください。")
    print("Stats APIはアクセス過多を検知して一時的にブロックすることがあります。")
else:
    run_results, score_history = result
    
    print(f"\n--- GameID: {target_game_id} の連続得点データ ---")
    if run_results.empty:
        print("指定された閾値以上の連続得点（Run）はありませんでした。")
    else:
        display(run_results)
    
    # グラフ描画
    print("\n--- モメンタムチャート ---")
    plot_pbp_flow(score_history, target_game_id)